# Assignment 2

## Formalia:

Please read the [assignment overview page](https://github.com/suneman/socialdata2026/wiki/Assignments) carefully before proceeding. This page contains information about formatting (including formats etc.), group sizes, and many other aspects of handing in the assignment. 

_If you fail to follow these simple instructions, it will negatively impact your grade!_

**Due date and time**: 
 - The assignment is due on Monday April 6th, 2026 at 23:55. 
 - Hand via DTU Learn. 
 - You should simply hand in the link to the github page with your short data story.

## A2: A short data story

This assignment is to create a short data-story based on the work we've done in class so far. In particular you will solve *Exercise 2.1* from *Week 8, Part 2*. The exercises for that week contain full details on how the story should be constructed.

You will need to think about how you place that page on your personal github page (there are several ways of doing that and it's not hard -- ask an LLM for help). At some point you'll also need to host your final project there, so keep that in mind too.

In [4]:
import pandas as pd
import plotly.express as px
import json
from urllib.request import urlopen

# 1. Load your data
# Using parse_dates ensures we can extract the year easily
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])

# 2. Extract Year and filter for PROSTITUTION
df['Year'] = df['Incident Date'].dt.year
df_filtered = df[df['Incident Category'] == 'PROSTITUTION'].copy()

# 3. Aggregate data by District and Year
# We count incidents per Police District per Year
map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')

# 4. Load the GeoJSON
url_geojson = 'https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson'
with urlopen(url_geojson) as response:
    sfpd_geojson = json.load(response)

# 5. Theme Colors from your CSS (Converted for Plotly)
bg_color = "#151719"      # Matches oklch(0.13 0.005 260)
primary_color = "#d97757" # Matches oklch(0.65 0.2 30) - Warm accent
text_color = "#f2f2f2"    # Off-white foreground
grid_color = "#333537"    # Subtle border color

# 6. Create the Animated Choropleth
fig = px.choropleth_map(
    map_data,
    geojson=sfpd_geojson,
    locations='Police District',
    featureidkey="properties.DISTRICT",
    color='Incidents',
    animation_frame='Year',
    # Gradient: From dark background to the warm primary accent
    color_continuous_scale=[[0, bg_color], [1, primary_color]],
    range_color=(0, map_data['Incidents'].max()),
    map_style="carto-darkmatter",
    zoom=11.5,
    center={"lat": 37.77, "lon": -122.42},
    opacity=0.8,
)

# 7. Layout & Aesthetic Styling
fig.update_layout(
    margin={"r":0,"t":80,"l":0,"b":40},
    paper_bgcolor=bg_color,
    plot_bgcolor=bg_color,
    font_color=text_color,
    font_family="Inter, system-ui, sans-serif",
    title={
        'text': "THE SPATIAL EVOLUTION<br><span style='font-size:14px; color:#656565;'>MIGRATION OF PROSTITUTION INCIDENTS IN SAN FRANCISCO</span>",
        'y': 0.95, 'x': 0.05,
        'font': {'size': 28, 'family': "Playfair Display, serif"}
    },
    # Style the colorbar (legend)
    coloraxis_colorbar=dict(
        title="INCIDENTS",
        thicknessmode="pixels", thickness=15,
        lenmode="fraction", len=0.6,
        yanchor="middle", y=0.5,
        ticks="outside"
    )
)

# Style the Slider and Play buttons to match the theme
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.update_layout(
    sliders=[{
        "currentvalue": {"prefix": "Year: ", "font": {"color": primary_color, "size": 20}},
        "bgcolor": grid_color,
        "activebgcolor": primary_color
    }]
)

# 8. Export to HTML (Frontend-ready)
fig.write_html("prostitution_map.html", include_plotlyjs='cdn', full_html=False)

In [15]:
import pandas as pd
import plotly.express as px
import json
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year
df_filtered = df[df['Incident Category'] == 'PROSTITUTION'].copy()

map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')

# ── GeoJSON ───────────────────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

# ── Theme ─────────────────────────────────────────────────────────────────────
BG      = "#151719"
PRIMARY = "#d97757"
TEXT    = "#f2f2f2"
GRID    = "#333537"

# ── Figure ────────────────────────────────────────────────────────────────────
fig = px.choropleth_map(
    map_data,
    geojson=sfpd_geojson,
    locations='Police District',
    featureidkey="properties.DISTRICT",
    color='Incidents',
    animation_frame='Year',
    color_continuous_scale=[[0, BG], [1, PRIMARY]],
    range_color=(0, map_data['Incidents'].max()),
    map_style="carto-darkmatter",
    zoom=11.5,
    center={"lat": 37.77, "lon": -122.42},
    opacity=0.8,
)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 80, "l": 0, "b": 40},
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    font_color=TEXT,
    font_family="Inter, system-ui, sans-serif",
    title={
        'text': "THE SPATIAL EVOLUTION<br><span style='font-size:14px; color:#656565;'>MIGRATION OF PROSTITUTION INCIDENTS IN SAN FRANCISCO</span>",
        'y': 0.95, 'x': 0.05,
        'font': {'size': 28, 'family': "Playfair Display, serif"},
    },
    coloraxis_colorbar=dict(
        title="INCIDENTS",
        thicknessmode="pixels", thickness=15,
        lenmode="fraction", len=0.6,
        yanchor="middle", y=0.5,
        ticks="outside",
    ),
)

# ── Fix slider — patch existing object, never replace it ─────────────────────
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

# Patch only the visual properties; leave steps/active intact
sl = fig.layout.sliders[0]
sl.update(
    bgcolor=GRID,
    activebgcolor=PRIMARY,
    bordercolor=GRID,
    tickcolor=TEXT,
    font=dict(color=TEXT, size=11),
    currentvalue=dict(
        prefix="Year: ",
        font=dict(color=PRIMARY, size=20),
        visible=True,
        xanchor="left",
    ),
    pad=dict(t=10, b=10),
)

# ── Export ────────────────────────────────────────────────────────────────────
fig.write_html("prostitution_map.html", include_plotlyjs='cdn', full_html=False)
print("✓ Saved → prostitution_map.html")

✓ Saved → prostitution_map.html


In [23]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')

# CRITICAL FIX 1: Sort by Year so the slider follows 2003 -> 2026 correctly
map_data = map_data.sort_values('Year')

# ── GeoJSON ───────────────────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

# ── Extract Centroids for Labels ──────────────────────────────────────────────
# We calculate the average lat/lon for each district to place a label in the center
district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    # Handle both Polygon and MultiPolygon
    coords = feature['geometry']['coordinates']
    all_points = []
    
    if feature['geometry']['type'] == 'Polygon':
        all_points = coords[0]
    else: # MultiPolygon
        for part in coords:
            all_points.extend(part[0])
            
    avg_lon = np.mean([p[0] for p in all_points])
    avg_lat = np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})

df_labels = pd.DataFrame(district_labels)

# ── Theme ─────────────────────────────────────────────────────────────────────
BG      = "#151719"
PRIMARY = "#d97757"
TEXT    = "#f2f2f2"
GRID    = "#333537"
BRIGHT_ORANGE = "#ff8c00" # New high-visibility orange

# ── Figure ────────────────────────────────────────────────────────────────────
# Base Choropleth
fig = px.choropleth_map(
    map_data,
    geojson=sfpd_geojson,
    locations='Police District',
    featureidkey="properties.DISTRICT",
    color='Incidents',
    animation_frame='Year',
    hover_name='Police District', # Shows district name on hover
    color_continuous_scale=[[0, BG], [1, PRIMARY]],
    range_color=(0, map_data['Incidents'].max()),
    map_style="carto-darkmatter",
    zoom=11.5,
    center={"lat": 37.77, "lon": -122.42},
    opacity=0.8,
)

# CRITICAL FIX 2: Add static text labels for districts
# We add this as a separate trace so names are always visible
fig.add_scattermap(
    lat=df_labels['lat'],
    lon=df_labels['lon'],
    mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=10, color= BRIGHT_ORANGE),
    showlegend=False,
    hoverinfo='skip'
)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 80, "l": 0, "b": 40},
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    font_color=TEXT,
    font_family="Inter, system-ui, sans-serif",
    title={
        'text': "THE SPATIAL EVOLUTION<br><span style='font-size:14px; color:#656565;'>MIGRATION OF PROSTITUTION INCIDENTS IN SAN FRANCISCO</span>",
        'y': 0.95, 'x': 0.05,
        'font': {'size': 28, 'family': "Playfair Display, serif"},
    },
    coloraxis_colorbar=dict(
        title="INCIDENTS",
        thicknessmode="pixels", thickness=15,
        lenmode="fraction", len=0.6,
        yanchor="middle", y=0.5,
        ticks="outside",
    ),
)

# ── Slider Fixes ─────────────────────────────────────────────────────────────
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

sl = fig.layout.sliders[0]
sl.update(
    bgcolor=GRID,
    activebgcolor=PRIMARY,
    bordercolor=GRID,
    tickcolor=TEXT,
    font=dict(color=TEXT, size=11),
    currentvalue=dict(
        prefix="Year: ",
        font=dict(color=PRIMARY, size=20),
        visible=True,
        xanchor="left",
    ),
    pad=dict(t=10, b=10),
)

# ── Export ────────────────────────────────────────────────────────────────────
fig.write_html("prostitution_map.html", include_plotlyjs='cdn', full_html=False)
print("✓ Saved → prostitution_map.html")

✓ Saved → prostitution_map.html


In [20]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── 1. Data Processing ────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# Filter: Prostitution only, exclude 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# Group for the Map
map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')
map_data = map_data.sort_values('Year')

# ── 2. Calculate Mission % for the React Counter ──────────────────────────────
# Total incidents in SF per year
total_yearly = map_data.groupby('Year')['Incidents'].sum().reset_index(name='Total')

# Mission incidents per year
mission_yearly = map_data[map_data['Police District'] == 'MISSION'].groupby('Year')['Incidents'].sum().reset_index(name='Mission_Total')

# Merge and calculate %
percentage_df = pd.merge(total_yearly, mission_yearly, on='Year', how='left').fillna(0)
percentage_df['Pct'] = (percentage_df['Mission_Total'] / percentage_df['Total'] * 100).round(1)

# Create a JS-friendly dictionary: { 2003: 12.5, 2004: 14.1, ... }
pct_lookup = percentage_df.set_index('Year')['Pct'].to_dict()

# ── 3. GeoJSON & Labels ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_pts = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    district_labels.append({'District': name, 'lat': np.mean([p[1] for p in all_pts]), 'lon': np.mean([p[0] for p in all_pts])})
df_labels = pd.DataFrame(district_labels)

# ── 4. Theme & Figure ─────────────────────────────────────────────────────────
BG, PRIMARY, TEXT, BRIGHT_ORANGE = "#151719", "#d97757", "#f2f2f2", "#ff8c00"

fig = px.choropleth_map(
    map_data, geojson=sfpd_geojson, locations='Police District',
    featureidkey="properties.DISTRICT", color='Incidents', animation_frame='Year',
    color_continuous_scale=[[0, BG], [1, PRIMARY]],
    range_color=(0, map_data['Incidents'].max()), map_style="carto-darkmatter",
    zoom=11.5, center={"lat": 37.77, "lon": -122.42}, opacity=0.8,
)

# Add District Labels
fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['District'], textfont=dict(size=10, color=BRIGHT_ORANGE),
    showlegend=False, hoverinfo='skip'
)

# ── 5. Layout Update (Requested Title Change) ─────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 60, "l": 0, "b": 40},
    paper_bgcolor=BG, plot_bgcolor=BG, font_color=TEXT,
    title={
        'text': "MIGRATION OF PROSTITUTION INCIDENTS IN SAN FRANCISCO",
        'y': 0.95, 'x': 0.05,
        'font': {'size': 18, 'color': '#656565', 'family': "Inter, sans-serif"},
    },
    coloraxis_colorbar=dict(title="INCIDENTS", thickness=15, len=0.5)
)

# Patch animation speeds
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

# ── 6. Export with the Data Bridge ───────────────────────────────────────────
# We export to HTML string first, then inject our custom JS
html_str = fig.to_html(include_plotlyjs='cdn', full_html=True)

bridge_script = f"""
<script>
    const pctLookup = {json.dumps(pct_lookup)};
    
    // Wait for Plotly to be ready
    window.addEventListener('load', function() {{
        const gd = document.getElementsByClassName('plotly-graph-div')[0];
        
        // Listen for frame changes
        gd.on('plotly_animated', function() {{
            const slider = gd.layout.sliders[0];
            const currentYear = slider.steps[slider.active].label;
            const currentPct = pctLookup[currentYear] || 0;
            
            // Send the data to your React/Next.js parent window
            window.parent.postMessage({{
                type: 'MAP_UPDATE',
                year: parseInt(currentYear),
                percentage: currentPct
            }}, '*');
        }});
    }});
</script>
</body>
"""

# Replace the closing body tag with our script + the closing tag
final_html = html_str.replace("</body>", bridge_script)

with open("prostitution_map.html", "w", encoding="utf-8") as f:
    f.write(final_html)

print("✓ Created: prostitution_map.html with React Bridge and Title update.")

✓ Created: prostitution_map.html with React Bridge and Title update.


In [22]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# 1. Filter: Prostitution only AND remove 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# 2. Group for Map
map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')
map_data = map_data.sort_values('Year')

# 3. Calculate Mission Percentage for the Data Bridge
total_per_year = map_data.groupby('Year')['Incidents'].sum().reset_index(name='Total')
mission_per_year = map_data[map_data['Police District'] == 'MISSION'].copy()
pct_df = pd.merge(total_per_year, mission_per_year, on='Year', how='left').fillna(0)
pct_df['Pct'] = (pct_df['Incidents'] / pct_df['Total'] * 100).round(1)
js_lookup = pct_df.set_index('Year')['Pct'].to_dict()

# ── GeoJSON & Centroids ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_points = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    avg_lon, avg_lat = np.mean([p[0] for p in all_points]), np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})
df_labels = pd.DataFrame(district_labels)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = px.choropleth_map(
    map_data, geojson=sfpd_geojson, locations='Police District',
    featureidkey="properties.DISTRICT", color='Incidents',
    animation_frame='Year', map_style="carto-darkmatter",
    color_continuous_scale=[[0, "#151719"], [1, "#d97757"]],
    zoom=11.5, center={"lat": 37.77, "lon": -122.42}, opacity=0.8
)

fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=11, color="#ff8c00")
)

# ── Unified Layout (Removing Internal Controls) ───────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    paper_bgcolor="#151719",
    font_color="#f2f2f2",
    title={
        'text': "MIGRATION OF PROSTITUTION INCIDENTS IN SAN FRANCISCO",
        'y': 0.98, 'x': 0.05, 'font': {'size': 16, 'color': '#656565'}
    },
    coloraxis_colorbar=dict(title="INCIDENTS", thickness=15, len=0.5),
)

# HIDE THE PLOTLY SLIDER AND PLAY BUTTON (Unified Control)
fig.layout.sliders[0].visible = False
fig.layout.updatemenus[0].visible = False

# ── The Data Bridge Script ────────────────────────────────────────────────────
# This allows React to say "Go to Year 2010" and Plotly will follow
bridge_script = f"""
<script>
    const percentages = {js_lookup};
    const gd = document.getElementsByClassName('plotly-graph-div')[0];

    window.addEventListener('message', function(event) {{
        if (event.data.type === 'SET_YEAR') {{
            const targetYear = event.data.year.toString();
            const steps = gd.layout.sliders[0].steps;
            const targetIndex = steps.findIndex(s => s.label === targetYear);
            
            if (targetIndex !== -1) {{
                Plotly.animate(gd, [steps[targetIndex].name], {{
                    mode: 'immediate',
                    frame: {{duration: 300, redraw: true}},
                    transition: {{duration: 300}}
                }});
                
                // Send percentage back to React
                window.parent.postMessage({{
                    type: 'UPDATE_PCT',
                    percentage: percentages[targetYear] || 0
                }}, '*');
            }}
        }}
    }});
</script>
"""

html_content = fig.to_html(include_plotlyjs='cdn', full_html=False)
with open("prostitution_map.html", "w") as f:
    f.write(html_content + bridge_script)
print("✓ Map updated with Data Bridge and no internal controls.")

✓ Map updated with Data Bridge and no internal controls.


In [24]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# 1. Filter: Prostitution only AND remove 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# 2. Group for Map
map_data = df_filtered.groupby(['Police District', 'Year']).size().reset_index(name='Incidents')
map_data = map_data.sort_values('Year')

# 3. Calculate Mission Percentage for the Data Bridge
total_per_year = map_data.groupby('Year')['Incidents'].sum().reset_index(name='Total')
mission_per_year = map_data[map_data['Police District'] == 'MISSION'].copy()
pct_df = pd.merge(total_per_year, mission_per_year, on='Year', how='left').fillna(0)
pct_df['Pct'] = (pct_df['Incidents'] / pct_df['Total'] * 100).round(1)
js_lookup = pct_df.set_index('Year')['Pct'].to_dict()

# ── GeoJSON & Centroids ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_points = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    avg_lon, avg_lat = np.mean([p[0] for p in all_points]), np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})
df_labels = pd.DataFrame(district_labels)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = px.choropleth_map(
    map_data, geojson=sfpd_geojson, locations='Police District',
    featureidkey="properties.DISTRICT", color='Incidents',
    animation_frame='Year', map_style="carto-darkmatter",
    color_continuous_scale=[[0, "#151719"], [1, "#d97757"]],
    zoom=11.5, center={"lat": 37.77, "lon": -122.42}, opacity=0.8
)

fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=11, color="#ff8c00")
)

# ── Clean Layout (No Title, No Controls) ─────────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 0, "l": 0, "b": 0}, # Full bleed
    paper_bgcolor="#151719",
    font_color="#f2f2f2",
    title=None, # Title REMOVED as requested
    coloraxis_colorbar=dict(title="INCIDENTS", thickness=15, len=0.5),
    showlegend=False
)

# REMOVE ALL INTERNAL UI (Stops auto-play and double-buttons)
fig.layout.updatemenus = []
fig.layout.sliders = []

# ── The Improved Data Bridge Script ──────────────────────────────────────────
# This version is passive: it does nothing until React sends a message.
bridge_script = f"""
<script>
    const percentages = {js_lookup};
    const gd = document.getElementsByClassName('plotly-graph-div')[0];

    window.addEventListener('message', function(event) {{
        if (event.data.type === 'SET_YEAR') {{
            const targetYear = event.data.year.toString();
            
            // Tell Plotly to jump to the frame named after the year
            Plotly.animate(gd, [targetYear], {{
                mode: 'immediate',
                frame: {{duration: 200, redraw: true}},
                transition: {{duration: 200}}
            }});
            
            // Send the Mission % for that year back to React
            window.parent.postMessage({{
                type: 'UPDATE_PCT',
                percentage: percentages[targetYear] || 0
            }}, '*');
        }}
    }});
</script>
"""

# Export
html_content = fig.to_html(include_plotlyjs='cdn', full_html=False)
with open("prostitution_map.html", "w") as f:
    f.write(html_content + bridge_script)
print("✓ Map updated. No title. No auto-play. Full React control active.")

✓ Map updated. No title. No auto-play. Full React control active.


In [25]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# 1. Filter: Prostitution only AND remove 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# 2. Ensure all Districts are present for every year (crucial for data-swapping)
districts = sorted(df_filtered['Police District'].unique())
years = sorted(df_filtered['Year'].unique())
all_combos = pd.MultiIndex.from_product([districts, years], names=['Police District', 'Year'])

map_data = df_filtered.groupby(['Police District', 'Year']).size().reindex(all_combos, fill_value=0).reset_index(name='Incidents')

# 3. Create a Data Lookup for JS (This is what makes it snappy)
# We store the incident counts for every year in a simple dictionary
data_lookup = {}
for y in years:
    data_lookup[str(y)] = map_data[map_data['Year'] == y]['Incidents'].tolist()

# 4. Calculate Mission Percentage
total_per_year = map_data.groupby('Year')['Incidents'].sum()
mission_per_year = map_data[map_data['Police District'] == 'MISSION'].set_index('Year')['Incidents']
pct_lookup = ((mission_per_year / total_per_year) * 100).round(1).fillna(0).to_dict()

# ── GeoJSON & Centroids ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_points = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    avg_lon, avg_lat = np.mean([p[0] for p in all_points]), np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})
df_labels = pd.DataFrame(district_labels).sort_values('Police District') # Match sorting

# ── Figure (Static Version) ──────────────────────────────────────────────────
# Note: We removed 'animation_frame'. This makes the map 100% static on load.
fig = px.choropleth_map(
    map_data[map_data['Year'] == 2003], # Initial year only
    geojson=sfpd_geojson, 
    locations='Police District',
    featureidkey="properties.DISTRICT", 
    color='Incidents',
    map_style="carto-darkmatter",
    color_continuous_scale=[[0, "#151719"], [1, "#d97757"]],
    range_color=(0, map_data['Incidents'].max()),
    zoom=11.5, center={"lat": 37.77, "lon": -122.42}, opacity=0.8
)

fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=11, color="#ff8c00")
)

# ── Clean Layout ──────────────────────────────────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    paper_bgcolor="#151719",
    font_color="#f2f2f2",
    title=None, # Removed all text as requested
    coloraxis_colorbar=dict(title="INCIDENTS", thickness=15, len=0.5),
)

# ── The Passive Bridge Script ────────────────────────────────────────────────
# This script does NOTHING until React sends a message. No auto-play.
bridge_script = f"""
<script>
    const allData = {data_lookup};
    const percentages = {pct_lookup};
    const gd = document.getElementsByClassName('plotly-graph-div')[0];

    window.addEventListener('message', function(event) {{
        if (event.data.type === 'SET_YEAR') {{
            const y = event.data.year.toString();
            if (allData[y]) {{
                // Directly update the data values (extremely fast)
                Plotly.restyle(gd, {{'z': [allData[y]]}}, [0]);
                
                // Send % back to React
                window.parent.postMessage({{
                    type: 'UPDATE_PCT',
                    percentage: percentages[y] || 0
                }}, '*');
            }}
        }}
    }});
</script>
"""

html_content = fig.to_html(include_plotlyjs='cdn', full_html=False)
with open("prostitution_map.html", "w") as f:
    f.write(html_content + bridge_script)
print("✓ Static map created. 0% auto-play risk. Controlled 100% by React.")

✓ Static map created. 0% auto-play risk. Controlled 100% by React.


In [28]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# 1. Filter: Prostitution only AND remove 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# 2. Ensure all Districts are present for every year (crucial for data-swapping)
districts = sorted(df_filtered['Police District'].unique())
years = sorted(df_filtered['Year'].unique())
all_combos = pd.MultiIndex.from_product([districts, years], names=['Police District', 'Year'])

map_data = df_filtered.groupby(['Police District', 'Year']).size().reindex(all_combos, fill_value=0).reset_index(name='Incidents')

# 3. Create a Data Lookup for JS (This is what makes it snappy)
# We store the incident counts for every year in a simple dictionary
data_lookup = {}
for y in years:
    data_lookup[str(y)] = map_data[map_data['Year'] == y]['Incidents'].tolist()

# 4. Calculate Mission Percentage
total_per_year = map_data.groupby('Year')['Incidents'].sum()
mission_per_year = map_data[map_data['Police District'] == 'MISSION'].set_index('Year')['Incidents']
pct_lookup = ((mission_per_year / total_per_year) * 100).round(1).fillna(0).to_dict()

# ── GeoJSON & Centroids ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_points = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    avg_lon, avg_lat = np.mean([p[0] for p in all_points]), np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})
df_labels = pd.DataFrame(district_labels).sort_values('Police District') # Match sorting

# ── Figure (Static Version) ──────────────────────────────────────────────────
# Note: We removed 'animation_frame'. This makes the map 100% static on load.
fig = px.choropleth_map(
    map_data[map_data['Year'] == 2003], # Initial year only
    geojson=sfpd_geojson, 
    locations='Police District',
    featureidkey="properties.DISTRICT", 
    color='Incidents',
    map_style="carto-darkmatter",
    color_continuous_scale=[[0, "#151719"], [1, "#d97757"]],
    range_color=(0, map_data['Incidents'].max()),
    zoom=11, center={"lat": 37.77, "lon": -122.42}, opacity=0.8
)

# CLEAN HOVER: Only show District and Incidents, remove coordinates/trace
fig.update_traces(
    hovertemplate="<b>%{location}</b><br>Incidents: %{z}<extra></extra>"
)

fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=11, color="#ff8c00"),
    hoverinfo='skip' # Labels won't trigger popups
)

# ── Clean Layout ──────────────────────────────────────────────────────────────
fig.update_layout(
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    paper_bgcolor="#151719",
    font_color="#f2f2f2",
    coloraxis_colorbar=dict(
        title="INCIDENTS", 
        thickness=15, 
        len=0.4, 
        y=0.5, # Centers legend vertically
        x=0.95 # Pushes legend to the edge
    ),
)
# ── The Passive Bridge Script ────────────────────────────────────────────────
# This script does NOTHING until React sends a message. No auto-play.
bridge_script = f"""
<script>
    const allData = {data_lookup};
    const percentages = {pct_lookup};
    const gd = document.getElementsByClassName('plotly-graph-div')[0];

    window.addEventListener('message', function(event) {{
        if (event.data.type === 'SET_YEAR') {{
            const y = event.data.year.toString();
            if (allData[y]) {{
                // Directly update the data values (extremely fast)
                Plotly.restyle(gd, {{'z': [allData[y]]}}, [0]);
                
                // Send % back to React
                window.parent.postMessage({{
                    type: 'UPDATE_PCT',
                    percentage: percentages[y] || 0
                }}, '*');
            }}
        }}
    }});
</script>
"""

html_content = fig.to_html(include_plotlyjs='cdn', full_html=False)
with open("prostitution_map.html", "w") as f:
    f.write(html_content + bridge_script)
print("✓ Static map created. 0% auto-play risk. Controlled 100% by React.")

✓ Static map created. 0% auto-play risk. Controlled 100% by React.


In [29]:
import pandas as pd
import plotly.express as px
import json
import numpy as np
from urllib.request import urlopen

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('sf_crime_merged_2003_2026.csv', parse_dates=['Incident Date'])
df['Year'] = df['Incident Date'].dt.year

# 1. Filter: Prostitution only AND remove 2026
df_filtered = df[(df['Incident Category'] == 'PROSTITUTION') & (df['Year'] < 2026)].copy()

# 2. Ensure all Districts are present for every year (crucial for data-swapping)
districts = sorted(df_filtered['Police District'].unique())
years = sorted(df_filtered['Year'].unique())
all_combos = pd.MultiIndex.from_product([districts, years], names=['Police District', 'Year'])

map_data = df_filtered.groupby(['Police District', 'Year']).size().reindex(all_combos, fill_value=0).reset_index(name='Incidents')

# 3. Create a Data Lookup for JS (This is what makes it snappy)
# We store the incident counts for every year in a simple dictionary
data_lookup = {}
for y in years:
    data_lookup[str(y)] = map_data[map_data['Year'] == y]['Incidents'].tolist()

# 4. Calculate Mission Percentage
total_per_year = map_data.groupby('Year')['Incidents'].sum()
mission_per_year = map_data[map_data['Police District'] == 'MISSION'].set_index('Year')['Incidents']
pct_lookup = ((mission_per_year / total_per_year) * 100).round(1).fillna(0).to_dict()

# ── GeoJSON & Centroids ───────────────────────────────────────────────────────
with urlopen('https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson') as r:
    sfpd_geojson = json.load(r)

district_labels = []
for feature in sfpd_geojson['features']:
    name = feature['properties']['DISTRICT']
    coords = feature['geometry']['coordinates']
    all_points = coords[0] if feature['geometry']['type'] == 'Polygon' else [p for part in coords for p in part[0]]
    avg_lon, avg_lat = np.mean([p[0] for p in all_points]), np.mean([p[1] for p in all_points])
    district_labels.append({'Police District': name, 'lat': avg_lat, 'lon': avg_lon})
df_labels = pd.DataFrame(district_labels).sort_values('Police District') # Match sorting

# ── Figure (Static Version) ──────────────────────────────────────────────────
# Note: We removed 'animation_frame'. This makes the map 100% static on load.
fig = px.choropleth_map(
    map_data[map_data['Year'] == 2003], # Initial year only
    geojson=sfpd_geojson, 
    locations='Police District',
    featureidkey="properties.DISTRICT", 
    color='Incidents',
    map_style="carto-darkmatter",
    color_continuous_scale=[[0, "#151719"], [1, "#d97757"]],
    range_color=(0, map_data['Incidents'].max()),
    zoom=11, center={"lat": 37.77, "lon": -122.42}, opacity=0.8
)

# CLEAN HOVER: Only show District and Incidents, remove coordinates/trace
fig.update_traces(
    hovertemplate="<b>%{location}</b><br>Incidents: %{z}<extra></extra>"
)

fig.add_scattermap(
    lat=df_labels['lat'], lon=df_labels['lon'], mode='text',
    text=df_labels['Police District'],
    textfont=dict(size=11, color="#ff8c00"),
    hoverinfo='skip' # Labels won't trigger popups
)

# ── Clean Layout Actualizado ──────────────────────────────────────────────────
fig.update_layout(
    # Añadimos un margen a la derecha (r: 100) para que la barra quepa fuera del mapa
    margin={"r": 100, "t": 0, "l": 0, "b": 0},
    paper_bgcolor="#151719",
    font_color="#f2f2f2",
    coloraxis_colorbar=dict(
        title="INCIDENTS", 
        thickness=20,          # Un poco más gruesa para que luzca mejor
        len=0.85,              # "Estirada": 0.85 significa que ocupa el 85% de la altura total
        y=0.5,                 # Centrada verticalmente
        x=1.02,                # Posición fuera del mapa (1.0 es el borde, 1.02 es justo afuera)
        xanchor="left",        # Anclamos la barra desde su lado izquierdo
        ypad=20,               # Espacio interno arriba y abajo
        tickfont=dict(size=12, color="#f2f2f2") # Fuente de los números
    ),
)
# ── The Passive Bridge Script ────────────────────────────────────────────────
# This script does NOTHING until React sends a message. No auto-play.
bridge_script = f"""
<script>
    const allData = {data_lookup};
    const percentages = {pct_lookup};
    const gd = document.getElementsByClassName('plotly-graph-div')[0];

    window.addEventListener('message', function(event) {{
        if (event.data.type === 'SET_YEAR') {{
            const y = event.data.year.toString();
            if (allData[y]) {{
                // Directly update the data values (extremely fast)
                Plotly.restyle(gd, {{'z': [allData[y]]}}, [0]);
                
                // Send % back to React
                window.parent.postMessage({{
                    type: 'UPDATE_PCT',
                    percentage: percentages[y] || 0
                }}, '*');
            }}
        }}
    }});
</script>
"""

html_content = fig.to_html(include_plotlyjs='cdn', full_html=False)
with open("prostitution_map.html", "w") as f:
    f.write(html_content + bridge_script)
print("✓ Static map created. 0% auto-play risk. Controlled 100% by React.")

✓ Static map created. 0% auto-play risk. Controlled 100% by React.
